# BFT65B FDIC early-warning pipeline - full run on Google Colab (7 Nov 2026)

Phạm Anh Chi (11230937), NEU. Repository: https://github.com/just4tk3009/Ph-m-Anh-Chi---11230937

Environment: Google Colab, Python 3.13.18, 2 CPUs, 12.7 GB RAM, Java 17 for PySpark; package versions in `requirements.txt`. Random seed: `quality.random_state = 42` in `config/config.yaml`. Data are not committed: they are rebuilt from the FDIC API.

How to read this notebook:

1. **Cell 1** downloads the data from the FDIC BankFind API (1984Q1-2026Q2, the download log below is the evidence for the download date) and ran a first `all`. Its Module 1-4 numbers are **superseded** by cell 2 (two data-quality fixes were made after it).
2. **Cell 2** is the run the report uses: it re-uses the downloaded Bronze files and runs every stage (silver -> gold -> benchmarks -> audit -> leakage -> features -> processing benchmark -> selection -> Freedman -> models -> MLlib -> facts).
3. **Cell 3** scores the newest quarter (2026Q2) with the frozen model.

All numbers in the report come from `reports/` written by cells 2-3.

## Cell 1 - download from the FDIC API (+ first, superseded run)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
W = '/content/drive/MyDrive/Phạm Anh Chi - 11230937'   # thư mục trên Drive
os.makedirs(W, exist_ok=True)
os.chdir('/content')
!rm -rf /content/Ph-m-Anh-Chi---11230937
!git clone -q https://github.com/just4tk3009/Ph-m-Anh-Chi---11230937.git
os.chdir('/content/Ph-m-Anh-Chi---11230937')
!pip install -q -r requirements.txt
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

D = f'--data-dir /content/data --reports-dir "{W}/reports"'
!python pipeline.py ingest {D}
!python pipeline.py all {D}
!cp -r /content/data/bronze "{W}/bronze_1984"
!cd "{W}" && rm -f reports.zip && zip -qr reports.zip reports
!pip freeze | grep -iE "^(polars|pyarrow|deltalake|pandas|numpy|scikit-learn|requests|urllib3|pyyaml|psutil|matplotlib|pytest|pyspark)=="

Mounted at /content/drive
2026-11-07 01:23:52 | INFO    | fdic_ews.ingestion.client | institutions filters=None offset=0 rows=10000/27835 (2.3s)
2026-11-07 01:23:54 | INFO    | fdic_ews.ingestion.client | institutions filters=None offset=10000 rows=20000/27835 (1.0s)
2026-11-07 01:23:55 | INFO    | fdic_ews.ingestion.client | institutions filters=None offset=20000 rows=27835/27835 (0.7s)
2026-11-07 01:23:55 | INFO    | fdic_ews.ingestion.client | failures filters=None offset=0 rows=4117/4117 (0.3s)
2026-11-07 01:23:55 | INFO    | fdic_ews.ingestion | Financials: 170 quarters requested (1984-03-31 .. 2026-06-30)
2026-11-07 01:24:00 | INFO    | fdic_ews.ingestion.client | financials filters=REPDTE:19840331 offset=0 rows=10000/17930 (5.0s)
2026-11-07 01:24:04 | INFO    | fdic_ews.ingestion.client | financials filters=REPDTE:19840331 offset=10000 rows=17930/17930 (3.1s)
2026-11-07 01:24:08 | INFO    | fdic_ews.ingestion.client | financials filters=REPDTE:19840630 offset=0 rows=10000/17935 

## Cell 2 - final run used in the report

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
W = '/content/drive/MyDrive/Phạm Anh Chi - 11230937'
os.chdir('/content')
!rm -rf /content/Ph-m-Anh-Chi---11230937 /content/data
!git clone -q https://github.com/just4tk3009/Ph-m-Anh-Chi---11230937.git
os.chdir('/content/Ph-m-Anh-Chi---11230937')
!pip install -q -r requirements.txt
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!mkdir -p /content/data && cp -r "{W}/bronze_1984" /content/data/bronze
D = f'--data-dir /content/data --reports-dir "{W}/reports"'
!python pipeline.py all {D}
!cd "{W}" && rm -f reports.zip && zip -qr reports.zip reports

Mounted at /content/drive
2026-11-07 12:06:55 | INFO    | fdic_ews.silver | silver financials: {'table': 'financials', 'bronze_files': 238, 'rows_bronze': 1682616, 'exact_duplicates_removed': 0, 'conflicting_key_duplicates': 0, 'rows_silver': 1682616, 'n_banks': 24064, 'n_quarters': 170, 'min_repdte': '1984-03-31', 'max_repdte': '2026-06-30', 'output': '/content/data/silver/financials.parquet'}
2026-11-07 12:06:58 | INFO    | fdic_ews.silver | silver institutions: {'table': 'institutions', 'rows_bronze': 27835, 'exact_duplicates_removed': 0, 'rows_silver': 27835, 'output': '/content/data/silver/institutions.parquet'}
2026-11-07 12:06:58 | INFO    | fdic_ews.silver | silver failures: {'table': 'failures', 'rows_bronze': 4117, 'exact_duplicates_removed': 0, 'rows_silver': 3036, 'assistance_rows_excluded': 593, 'rows_without_cert': 485, 'repeated_cert_rows': 3, 'min_faildate': '1970-02-22', 'max_faildate': '2026-07-17', 'output': '/content/data/silver/failures.parquet'}
2026-11-07 12:07:0

## Cell 3 - score the newest quarter with the frozen model

In [3]:
!git pull -q && python pipeline.py predict {D}
!cd "{W}" && rm -f reports.zip && zip -qr reports.zip reports

2026-11-07 16:39:39 | INFO    | fdic_ews.modeling.predict | predict: {'scored_at': '2026-11-07T16:39:39Z', 'quarter': '2026-06-30', 'banks_scored': 4313, 'model': 'rf_weights', 'threshold': 0.05478758353509188, 'alerts_at_threshold': 4, 'alerts_top5pct': 216, 'mean_prob_pct': 0.03977355768545212}
{
  "scored_at": "2026-11-07T16:39:39Z",
  "quarter": "2026-06-30",
  "banks_scored": 4313,
  "model": "rf_weights",
  "threshold": 0.05478758353509188,
  "alerts_at_threshold": 4,
  "alerts_top5pct": 216,
  "mean_prob_pct": 0.03977355768545212
}
2026-11-07 16:39:39 | INFO    | fdic_ews.cli | stage predict finished in 9.5s
